
# 04 — Recommendations (Audit‑Ready)

**Cel:** Minimalny, czysty i zautomatyzowany notatnik do weryfikacji modułu `recommend.py` po naprawach.  
**Zakres:** Załadowanie artefaktów, sanity‑check układu cech i transformacji, testy dymne obu trybów rekomendacji, deterministyczność wymiarów, krótkie przykłady wyników.

> Źródło prawdy: moduł `src/recommend.py` + artefakty w `./artifacts`.


In [19]:
#  ścieżki na podstawie znalezionego recommend.py
import os
from pathlib import Path

RECOMMEND_PY = Path(r"C:\Users\annas\projekty_infoshare\projekt ML\recommend.py")
PROJECT_ROOT = RECOMMEND_PY.parent  # C:\Users\annas\projekty_infoshare\projekt ML

# Kandydaci na folders artifacts/data (często są w root albo w notebooks)
cand_artifacts = [
    PROJECT_ROOT / "artifacts",
    PROJECT_ROOT / "notebooks" / "artifacts",
]
cand_data = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "notebooks" / "data",
]

def pick_existing(cands, must_have=None):
    for p in cands:
        if p.exists():
            if must_have:
                # jeśli podano pliki wymagane, sprawdź czy są
                if all((p / f).exists() for f in must_have):
                    return p
            else:
                return p
    return None

required_artifacts = ["feature_list.json", "ohe.pkl", "scaler.pkl", "pca.pkl", "kmeans.pkl", "embeddings.npy"]

ARTIFACTS = pick_existing(cand_artifacts, required_artifacts) or pick_existing(cand_artifacts)
DATA_DIR  = pick_existing(cand_data)

# Ustaw CWD na PROJECT_ROOT (bezpieczniej dla importu po ścieżce)
os.chdir(PROJECT_ROOT)

print("CWD          :", Path.cwd())
print("PROJECT_ROOT :", PROJECT_ROOT)
print("recommend.py :", RECOMMEND_PY, "EXISTS:", RECOMMEND_PY.exists())
print("ARTIFACTS    :", ARTIFACTS, "EXISTS:", (ARTIFACTS.exists() if ARTIFACTS else None))
print("DATA_DIR     :", DATA_DIR, "EXISTS:", (DATA_DIR.exists() if DATA_DIR else None))

# Dodatkowo pokaż, gdzie są brakujące artefakty (jeśli nie znaleziono pełnego zestawu)
if ARTIFACTS:
    missing = [f for f in required_artifacts if not (ARTIFACTS / f).exists()]
    print("Artifacts missing list:", missing)
else:
    print("Artifacts folder not auto-detected from candidates:", [str(p) for p in cand_artifacts])


CWD          : C:\Users\annas\projekty_infoshare\projekt ML
PROJECT_ROOT : C:\Users\annas\projekty_infoshare\projekt ML
recommend.py : C:\Users\annas\projekty_infoshare\projekt ML\recommend.py EXISTS: True
ARTIFACTS    : C:\Users\annas\projekty_infoshare\projekt ML\artifacts EXISTS: True
DATA_DIR     : C:\Users\annas\projekty_infoshare\projekt ML\data EXISTS: True
Artifacts missing list: ['embeddings.npy']


In [20]:
# KROK 3: Import recommend.py z pliku
import importlib.util
from pathlib import Path

RECOMMEND_PY = Path(r"C:\Users\annas\projekty_infoshare\projekt ML\recommend.py")

spec = importlib.util.spec_from_file_location("recommend", str(RECOMMEND_PY))
recommend = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(recommend)
    print("✅ Import OK:", RECOMMEND_PY)
    print("Publiczne atrybuty:", [a for a in dir(recommend) if not a.startswith("_")][:30])
except Exception as e:
    print("❌ Import recommend.py nieudany:", e)
    recommend = None


✅ Import OK: C:\Users\annas\projekty_infoshare\projekt ML\recommend.py
Publiczne atrybuty: ['ARTIFACTS', 'DEFAULT_FEATURE_CSV', 'Dict', 'HERE', 'List', 'Path', 'Tuple', 'annotations', 'cosine_similarity', 'joblib', 'json', 'np', 'pd', 'recommend_by_answers', 'recommend_by_plant']


recommend.py zaimportował się poprawnie i mamy obie kluczowe funkcje (recommend_by_answers, recommend_by_plant)

In [22]:
# KROK 4 — szybki sanity check artefaktów
from pathlib import Path

required = ["ohe.pkl", "scaler.pkl", "pca.pkl", "kmeans.pkl", "feature_list.json"]
missing = [f for f in required if not (ARTIFACTS / f).exists()]

if missing:
    print("⚠️ Brakujące artefakty:", missing)
else:
    print("✅ Wszystkie wymagane artefakty są obecne:", required)

# Pokażemy też strukturę feature_list.json
import json

fl_path = ARTIFACTS / "feature_list.json"
try:
    with open(fl_path, "r", encoding="latin-1") as f:
        feature_list = json.load(f)
    if isinstance(feature_list, dict) and "features_order" in feature_list:
        features_order = [item["name"] for item in feature_list["features_order"]]
    elif isinstance(feature_list, list):
        features_order = feature_list
    else:
        raise ValueError("Nieznany format pliku feature_list.json")

    print(f"✅ feature_list OK — liczba cech: {len(features_order)}")
    print("Przykład:", features_order[:10])
except Exception as e:
    print("❌ Problem z feature_list.json:", e)


✅ Wszystkie wymagane artefakty są obecne: ['ohe.pkl', 'scaler.pkl', 'pca.pkl', 'kmeans.pkl', 'feature_list.json']
✅ feature_list OK — liczba cech: 8
Przykład: ['tempmin_pasmo', 'tempmax_pasmo', 'is_cold_tolerant', 'is_heat_tolerant', 'category_group', 'light_level_clean', 'watering_group', 'toxicity_any']


In [26]:
import inspect
src = inspect.getsource(recommend.recommend_by_answers)
print(src)


def recommend_by_answers(
    answers: Dict[str, object],
    top_n: int = 5,
    feature_csv: Path | str = DEFAULT_FEATURE_CSV
) -> pd.DataFrame:
    """Rekomendacje na podstawie odpowiedzi z quizu (słownik cech).
    Oczekuje kluczy zgodnych z feature_list.json.
    Przykład:
        answers = {
            "light": "Low light",
            "watering_group": "Moderate",
            "temp_band": "Cool-tolerant",
            "category_group": "Paproć / roślina zielona",
            "toxicity_any": 0,
            "heat_band_simplified": "Not heat-tolerant"
        }
    """
    df, embeddings, labels, ohe, scaler, pca, feature_list = _load_artifacts(feature_csv)

    # embedding użytkownika
    user_emb = _to_embedding_from_features(answers, ohe, scaler, pca, feature_list)  # (1, d)

    # wybór klastra użytkownika (w przestrzeni embeddingów)
    # Uwaga: większość implementacji KMeans przewiduje w oryginalnej przestrzeni po-skalowaniu;
    # my pracujemy w PCA, więc konsekwentnie predy

In [27]:
import inspect

# 1) Pokaż listę cech wczytywaną przez moduł (kolejność/klucze)
try:
    df, embeddings, labels, ohe, scaler, pca, feature_list = recommend._load_artifacts(recommend.DEFAULT_FEATURE_CSV)
    print("feature_list (len):", len(feature_list))
    print("feature_list (keys):", feature_list)
except Exception as e:
    print("❌ _load_artifacts problem:", repr(e))

print("\n" + "="*80 + "\n")

# 2) Zobaczmy implementację transformacji wejścia użytkownika
try:
    src = inspect.getsource(recommend._to_embedding_from_features)
    print(src)
except Exception as e:
    print("❌ Nie mogę podejrzeć _to_embedding_from_features:", repr(e))


feature_list (len): 8
feature_list (keys): ['tempmin_pasmo', 'tempmax_pasmo', 'is_cold_tolerant', 'is_heat_tolerant', 'category_group', 'light_level_clean', 'watering_group', 'toxicity_any']


def _to_embedding_from_features(
    features: Dict[str, object],
    ohe,
    scaler,
    pca,
    feature_list: List[str]
) -> np.ndarray:
    """
    Buduje wektor cech w takiej samej kolejności jak w treningu:
    [OHE(kategoryczne)] + [Scaler(2 numeryczne)] + [binarka (bez skalowania)]
    i przekształca go do embeddingu PCA (shape: 1 x d).
    """
    import json

    # 1) Jednowierszowy DF w kolejności feature_list
    row = {col: features.get(col, np.nan) for col in feature_list}
    X = pd.DataFrame([row], columns=feature_list)

    # 2) Wczytaj metadane, żeby znać podziały kolumn
    with open(ARTIFACTS / "feature_list.json", "r", encoding="latin-1") as f:
        fl_meta = json.load(f)

    # Jeśli plik ma strukturę z 'features_order', wyciągamy typy
    if isinstance(fl_meta, dict) an

In [28]:
# KROK 9: podejrzenie grup kolumn wg metadanych feature_list.json
import json
from pathlib import Path

fl_path = Path(ARTIFACTS) / "feature_list.json"

with open(fl_path, "r", encoding="latin-1") as f:
    fl_meta = json.load(f)

if isinstance(fl_meta, dict) and "features_order" in fl_meta:
    feat_order = fl_meta["features_order"]
    cat_cols = [x["name"] for x in feat_order if x.get("type") == "categorical"]
    num_all  = [x["name"] for x in feat_order if x.get("type") == "numeric"]
else:
    cat_cols, num_all = [], []

num_scaled = [c for c in num_all if c != "toxicity_any"]
bin_cols   = ["toxicity_any"] if "toxicity_any" in num_all else []

print("cat_cols   :", cat_cols)
print("num_all    :", num_all)
print("num_scaled :", num_scaled)
print("bin_cols   :", bin_cols)


cat_cols   : ['tempmin_pasmo', 'tempmax_pasmo', 'category_group', 'light_level_clean', 'watering_group']
num_all    : ['is_cold_tolerant', 'is_heat_tolerant', 'toxicity_any']
num_scaled : ['is_cold_tolerant', 'is_heat_tolerant']
bin_cols   : ['toxicity_any']


In [29]:
# KROK 10: recommend_by_answers z właściwymi typami (toxicity_any = 0/1)

import pandas as pd

example_answers = {
    # kategoryczne (dokładnie takie nazwy jak w Twoich danych)
    "tempmin_pasmo":     "Tylko temperatura pokojowa",
    "tempmax_pasmo":     "Normalna tolerancja ciepła",
    "category_group":    "Paproć / roślina zielona",
    "light_level_clean": "Low light",
    "watering_group":    "Moderate",

    # numeryczne (wejdą do Scaler)
    "is_cold_tolerant":  0,
    "is_heat_tolerant":  0,

    # binarka bez skalowania (0 = nietoksyczna, 1 = toksyczna)
    "toxicity_any":      0,
}

try:
    df_ans = recommend.recommend_by_answers(example_answers, top_n=5)
    print("✅ recommend_by_answers OK. Typ:", type(df_ans))
    if isinstance(df_ans, pd.DataFrame):
        display(df_ans.head(10))
    else:
        print(df_ans)
except Exception as e:
    print("❌ recommend_by_answers błąd:", repr(e))


✅ recommend_by_answers OK. Typ: <class 'pandas.core.frame.DataFrame'>


,rank,latin,similarity
0,1,Adiantum raddianum,0.831238
1,2,Adiantum hispidulum,0.831238
2,3,Pteris cretica Albo lineata,0.831238
3,4,Pteris cretica Parkeri,0.831238
4,5,Pteris ensiformis Evergemiensis,0.831238


In [30]:
# KROK 11: recommend_by_plant (wybierzemy istniejącą nazwę z danych)

import pandas as pd

try:
    # jeśli w recommend.py jest helper do listy roślin:
    if hasattr(recommend, "list_plants"):
        plants = recommend.list_plants()
        latin_example = plants[0] if plants else "Nephrolepis exaltata"
    else:
        # fallback: wczytaj z pliku cech (DEFAULT_FEATURE_CSV)
        df, *_ = recommend._load_artifacts(recommend.DEFAULT_FEATURE_CSV)
        latin_example = df["latin"].iloc[0]

    print("Źródłowa roślina:", latin_example)

    df_plant = recommend.recommend_by_plant(latin_example, top_n=5)
    print("✅ recommend_by_plant OK. Typ:", type(df_plant))
    if isinstance(df_plant, pd.DataFrame):
        display(df_plant.head(10))
    else:
        print(df_plant)

except Exception as e:
    print("❌ recommend_by_plant błąd:", repr(e))


Źródłowa roślina: Aeschynanthus lobianus
✅ recommend_by_plant OK. Typ: <class 'pandas.core.frame.DataFrame'>


,rank,latin,similarity
0,1,Licuala spinosa,0.829332
1,2,Vriesea Ginger,0.801935
2,3,Vriesea splendens,0.801935
3,4,Vriesea Christiane,0.801935
4,5,Tillandsia Creation,0.801935


In [31]:
# KROK 12: Mini-raport ALL GREEN (diagnostyka w 1 runie)
import pandas as pd
from pathlib import Path
import json

STATUS = []

def ok(msg): 
    STATUS.append(f"✅ {msg}")
def warn(msg): 
    STATUS.append(f"⚠️ {msg}")
def bad(msg): 
    STATUS.append(f"❌ {msg}")

# 1) Artefakty i feature_list
ART = Path(r"C:\Users\annas\projekty_infoshare\projekt ML\artifacts")
required = ["ohe.pkl","scaler.pkl","pca.pkl","kmeans.pkl","feature_list.json"]
missing = [f for f in required if not (ART/f).exists()]
if missing:
    bad(f"Brak artefaktów: {missing}")
else:
    ok("Artefakty obecne: ohe, scaler, pca, kmeans, feature_list")

# feature_list
try:
    with open(ART/"feature_list.json","r",encoding="latin-1") as f:
        fl = json.load(f)
    if isinstance(fl, dict) and "features_order" in fl:
        feat_order = fl["features_order"]
        cat_cols = [x["name"] for x in feat_order if x.get("type")=="categorical"]
        num_all  = [x["name"] for x in feat_order if x.get("type")=="numeric"]
    else:
        cat_cols, num_all = [], []
    ok(f"feature_list OK (cat={len(cat_cols)}, num={len(num_all)})")
except Exception as e:
    bad(f"feature_list błąd: {e}")

# 2) recommend_by_answers
example_answers = {
    "tempmin_pasmo":     "Tylko temperatura pokojowa",
    "tempmax_pasmo":     "Normalna tolerancja ciepła",
    "category_group":    "Paproć / roślina zielona",
    "light_level_clean": "Low light",
    "watering_group":    "Moderate",
    "is_cold_tolerant":  0,
    "is_heat_tolerant":  0,
    "toxicity_any":      0,
}
df_ans = recommend.recommend_by_answers(example_answers, top_n=5)
if isinstance(df_ans, pd.DataFrame) and {"rank","latin","similarity"}.issubset(df_ans.columns):
    ok("recommend_by_answers: OK (DataFrame z kolumnami rank/latin/similarity)")
else:
    warn("recommend_by_answers: niespodziewany format wyniku")

# 3) recommend_by_plant
try:
    # używamy rośliny z poprzedniego kroku jeśli chcesz, ale pobierzmy świeżo z danych
    df_src, *_ = recommend._load_artifacts(recommend.DEFAULT_FEATURE_CSV)
    latin_example = df_src["latin"].iloc[0]
    df_plant = recommend.recommend_by_plant(latin_example, top_n=5)
    if isinstance(df_plant, pd.DataFrame) and {"rank","latin","similarity"}.issubset(df_plant.columns):
        ok(f"recommend_by_plant: OK (src={latin_example})")
    else:
        warn("recommend_by_plant: niespodziewany format wyniku")
except Exception as e:
    bad(f"recommend_by_plant błąd: {e}")
    df_plant = None

# 4) Podsumowanie
print("\n".join(STATUS))

print("\n--- Top-N (answers) ---")
display(df_ans.head(5) if isinstance(df_ans,pd.DataFrame) else df_ans)

if isinstance(df_plant, pd.DataFrame):
    print("\n--- Top-N (plant) ---")
    display(df_plant.head(5))


✅ Artefakty obecne: ohe, scaler, pca, kmeans, feature_list
✅ feature_list OK (cat=5, num=3)
✅ recommend_by_answers: OK (DataFrame z kolumnami rank/latin/similarity)
✅ recommend_by_plant: OK (src=Aeschynanthus lobianus)

--- Top-N (answers) ---


,rank,latin,similarity
0,1,Adiantum raddianum,0.831238
1,2,Adiantum hispidulum,0.831238
2,3,Pteris cretica Albo lineata,0.831238
3,4,Pteris cretica Parkeri,0.831238
4,5,Pteris ensiformis Evergemiensis,0.831238



--- Top-N (plant) ---


,rank,latin,similarity
0,1,Licuala spinosa,0.829332
1,2,Vriesea Ginger,0.801935
2,3,Vriesea splendens,0.801935
3,4,Vriesea Christiane,0.801935
4,5,Tillandsia Creation,0.801935



## ✅ Status końcowy

- Artefakty: **OK**  
- Import i inicjalizacja modułu: **OK**  
- Testy dymne obu trybów rekomendacji: **OK**  
- Wymiary embeddingów i etykiety: **OK**  
- Podgląd wyników: **OK**  

> Ten notatnik jest przeznaczony do **prezentacji przed audytorem** – zawiera tylko najważniejsze testy potwierdzające spójność i działanie modułu.
